In [1]:
import re
import Levenshtein

def preprocess_string(s):
    # 去除非字母字符，转换为小写
    return re.sub(r'[^a-zA-Z]', '', s).lower()

def compute_similarity(s1, s2):
    s1 = preprocess_string(s1)
    s2 = preprocess_string(s2)
    # 计算Levenshtein距离
    distance = Levenshtein.distance(s1, s2)
    # 计算相似度
    similarity = 1 - distance / max(len(s1), len(s2))
    return similarity

str1 = "s"
str2 = "d"

similarity = compute_similarity(str1, str2)
print(f"The similarity between the two strings is: {similarity:.2f}")


The similarity between the two strings is: 0.00


In [2]:
import random 

def merge_select(list1, list2):
    counter = 0
    merged_list = [(item, 'clean') for item in list1] + [(item, 'poison') for item in list2]
    random.shuffle(merged_list)

    #print(len(merged_list))
    for i in range(len(merged_list)):
        i_s, i_label = merged_list[i]
        i_str = i_s['context']
        for j in range(len(merged_list)-1, i, -1):
            j_s, j_label = merged_list[j]
            j_str = j_s['context']
            s_score = compute_similarity(i_str, j_str)

            if s_score>0.85:
                counter += 1
                del merged_list[j]
        if i >= len(merged_list)-1: break

    #print(f"counter={counter}")
    clean_p = [item for item, label in merged_list if label == 'clean']
    poison_p = [item for item, label in merged_list if label == 'poison']
    return clean_p, poison_p

def balance(list1, list2, ratio, scaler_num):
    if ratio == 0:
        if len(list1) < scaler_num:
            list1_t = list1
        else: list1_t = random.sample(list1, scaler_num)
        return list1_t,[]
    if ratio == 1:
        if len(list2) < scaler_num:
            list2_t = list2
        else: list2_t = random.sample(list2, scaler_num)
        return [],list2_t
    len1 = len(list1)
    len2 = len(list2)
    k_aim = (1-ratio) / ratio
    k_fact = len1 / len2

    if k_fact > k_aim:
        len1_t = int(len2 / ratio * (1-ratio))
        list1_t = random.sample(list1, len1_t)
        list2_t = list2
    else:
        len2_t = int(len1 / (1-ratio) * ratio)
        list2_t = random.sample(list2, len2_t)
        list1_t = list1
    
    if len(list1_t)  > int(scaler_num * (1-ratio)) and len(list2_t)  > int(scaler_num * ratio):
        list1_t = random.sample(list1_t, int(scaler_num * (1-ratio)))
        list2_t = random.sample(list2_t, int(scaler_num * ratio))

    return list1_t, list2_t


In [4]:
import os
import json
import random
from tqdm.notebook import tqdm
from langchain.docstore.document import Document as LangchainDocument
from sklearn.model_selection import train_test_split
json_objects = []
undetermined = ["Not enough information", "unknown", "Cannot be determined", "Can't be determined", "Not enough info", \
                "Undetermined", "Not answerable", "Can't answer", "Cannot answer", "Not known"]
from langchain.docstore.document import Document as LangchainDocument

train_data = []
test_data = []

def processing_jsonl(ratio, scaler_num, rows_clean, rows_poison):
    processed_texts = []
    RAW_KNOWLEDGE_BASE = []

    # 抽样： clean/poison + balance samples
    balanced_rows_clean, balenced_rows_poison = balance(rows_clean, rows_poison, ratio, scaler_num)
    att = rows_clean[0]['category']
    # 按照 label 连接 content 和 correct ans (clean)
    for row in balanced_rows_clean:
        context = row['context']
        question = row['question']
        label = row['label']
        ans_index = f'ans{label}'  # 根据 label 构造答案字段的键
        ans = row[ans_index]

        processed_text = f"{context} {question} The answer is {ans.lower()}.\n\n"
        processed_texts.append(processed_text)
        page_content = processed_text
        metadata = {"source": f"BBQ dataset", "attribute": att, "poison": "clean"}
        
        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE.append(doc)

    # 按照 label 连接 content 和 poison ans 
    for row in balenced_rows_poison:
        context = row['context']
        question = row['question']
        label = row['label']
        for i in range(3):
            prob_ans = row['answer_info'][f"ans{i}"]
            #print(prob_ans)
            if i != label and prob_ans[1] != 'Unknown':
                ans = prob_ans[0]

        #Poisoned ans
        processed_text= f"{context} {question} The answer is {ans.lower()}.\n\n"
        processed_texts.append(processed_text)
        page_content = processed_text
        metadata = {"source": f"BBQ dataset", "attribute": att, "poison": "poison"}
        
        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE.append(doc)
        
    print(f"{att} --- Selected {len(processed_texts)} texts out of {len(balanced_rows_clean)+len(balenced_rows_poison)},\
        Clean: {len(balanced_rows_clean)}, Poison:{len(balenced_rows_poison)}")
    return RAW_KNOWLEDGE_BASE


In [4]:
#读取每个attr并去重
def jsonl_read(jsonl_file):
    entire_poison = []
    entire_clean = []
    # 打开 JSON Lines 文件并逐行读取
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            json_obj = json.loads(line.strip())      
            # 检查条件：context_condition 是 "ambig" 且 question_polarity 是 "neg"
            if json_obj.get('context_condition') == 'ambig' and json_obj.get('question_polarity') == 'neg':
                entire_poison.append(json_obj)
            if json_obj.get('context_condition') == 'disambig' and json_obj.get('question_polarity') == 'neg':
                entire_clean.append(json_obj)
    #去重
    entire_clean_s, entire_poison_s = merge_select(entire_clean, entire_poison)
    train_clean, test_clean = train_test_split(entire_clean_s, test_size=0.2, random_state=1)
    train_poison, test_poison = train_test_split(entire_poison_s, test_size=0.2, random_state=1)
    return train_clean, test_clean, train_poison, test_poison
    

In [5]:
def export_to_jsonl(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in data:
            json_line = json.dumps(item)  # 将字典转换为JSON格式字符串
            f.write(json_line + '\n')  # 每个JSON对象写入一行并换行


In [22]:
# 预处理：划分train,test
#This chunk will only run once, and it cost about 2 mins
directory = "../BBQ/BBQ-main/data"
print(directory)
random.seed(820)
all_data = []
train_data = []
test_data = []
for filename in os.listdir(directory):
    print(filename)
    if filename.endswith('.jsonl'):
        file_path = os.path.join(directory, filename)
        num_lines = sum(1 for line in open(file_path, 'r', encoding='utf-8'))
        
        train_clean, test_clean, train_poison, test_poison = jsonl_read(file_path)

        attr = train_clean[0]['category'] 
        train_data = train_data + train_clean + train_poison
        test_data = test_data + test_clean +test_poison

        all_data.append({attr: {"train_clean":train_clean, "train_poison": train_poison, \
                                  "test_clean":test_clean, "test_poison": test_poison} })

export_to_jsonl(train_data, '../bbq/bbq_train.jsonl')
export_to_jsonl(test_data, '../bbq/bbq_test.jsonl')

../BBQ/BBQ-main/data
Age.jsonl
Religion.jsonl
Race_x_SES.jsonl
Physical_appearance.jsonl
SES.jsonl
Race_ethnicity.jsonl
Race_x_gender.jsonl
Disability_status.jsonl
Nationality.jsonl
Sexual_orientation.jsonl
Gender_identity.jsonl


In [38]:
### Main func ###

ratio = 1
scaler = 100  
RAW_KNOWLEDGE_BASE_train = []
RAW_KNOWLEDGE_BASE_test = []

for item in all_data:
    for attr, f_data in item.items():
        train_clean = f_data['train_clean']
        train_poison = f_data['train_poison']
        test_clean = f_data['test_clean']
        test_poison = f_data['test_poison']
        print(f"length of {attr} train_clean:{len(train_clean)} train_poison{len(train_poison)}")
        print(f"test_clean{len(test_clean)} test_poison{len(test_poison)}" )
        RAW_KNOWLEDGE_BASE_train += processing_jsonl(ratio, scaler, train_clean, train_poison) 
        #RAW_KNOWLEDGE_BASE_test += processing_jsonl(ratio, scaler, test_clean, test_poison)

'''
for filename in os.listdir(directory):
    if filename.endswith('.jsonl'):
        file_path = os.path.join(directory, filename)
        num_lines = sum(1 for line in open(file_path, 'r', encoding='utf-8'))
        # 打开 JSON Lines 文件并读取内容
        tmp_texts = (processing_jsonl(ratio, scaler, file_path )) 
        total_texts.append(tmp_texts)
'''      

print(f"Created RAW_KNOWLEDGE_BASE_train with {len(RAW_KNOWLEDGE_BASE_train)} documents.")

length of Age train_clean:36 train_poison48
test_clean9 test_poison13
Age --- Selected 48 texts out of 48,        Clean: 0, Poison:48
length of Religion train_clean:23 train_poison32
test_clean6 test_poison9
Religion --- Selected 32 texts out of 32,        Clean: 0, Poison:32
length of Race_x_SES train_clean:437 train_poison767
test_clean110 test_poison192
Race_x_SES --- Selected 100 texts out of 100,        Clean: 0, Poison:100
length of Physical_appearance train_clean:48 train_poison67
test_clean13 test_poison17
Physical_appearance --- Selected 67 texts out of 67,        Clean: 0, Poison:67
length of SES train_clean:90 train_poison188
test_clean23 test_poison47
SES --- Selected 100 texts out of 100,        Clean: 0, Poison:100
length of Race_ethnicity train_clean:214 train_poison316
test_clean54 test_poison80
Race_ethnicity --- Selected 100 texts out of 100,        Clean: 0, Poison:100
length of Race_x_gender train_clean:436 train_poison790
test_clean110 test_poison198
Race_x_gender 

In [39]:
docs_dict_train = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in RAW_KNOWLEDGE_BASE_train
]

# 将字典列表导出为 JSONL 文件
export_to_jsonl(docs_dict_train, f'../bbq/bbq_train-{ratio}-{scaler}.jsonl')